# Colab / Kaggle runner - Fake Review Detection

Runs the project on a free **T4 GPU**. Work survives session timeouts because `data/` and `checkpoints/`
live on **Google Drive** (symlinked into the repo) and `results/` is synced to Drive after every step.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Steps: 1) GPU check → 2) settings → 3) mount Drive → 4) get the code → 5) install → 6) link Drive folders →
7) download data (once) → 8) tests + smoke test → 9) full runs → 10) save results.

In [ ]:
!nvidia-smi -L || echo 'No GPU - select Runtime > Change runtime type > T4 GPU'

### Settings
* `REPO_URL`: your GitHub repo. For a **private** repo, add a Colab secret (🔑 icon on the left) called
  `GITHUB_TOKEN` holding a fine-grained token with read access.
* No GitHub? Zip the project folder, upload it to `DRIVE_DIR` as `fake-review-detection.zip`, and leave `REPO_URL` empty.

In [ ]:
REPO_URL = "https://github.com/plmcodes/fake-review-detection.git"   # <- change if needed ("" = use the zip on Drive)
BRANCH = "main"
DRIVE_DIR = "/content/drive/MyDrive/fake-review-detection"   # data, checkpoints and results are kept here
REPO_DIR = "/content/fake-review-detection"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
import os, subprocess
if os.path.exists(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
elif REPO_URL:
    url = REPO_URL
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{token}@")
    except Exception:
        pass  # public repo, or no token secret set
    subprocess.run(["git", "clone", "-b", BRANCH, url, REPO_DIR], check=True)
else:
    subprocess.run(["unzip", "-q", "-o", f"{DRIVE_DIR}/fake-review-detection.zip", "-d", "/content"], check=True)
%cd {REPO_DIR}
!git log --oneline -3 || true

Colab already ships a CUDA build of PyTorch, so `requirements.txt` deliberately does **not** pin torch.

In [ ]:
!pip install -q -r requirements.txt
import torch; print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# data/ and checkpoints/ -> symlinks to Drive (survive timeouts). results/ is tracked in git, so we sync it instead.
import os, shutil
for sub in ["data", "checkpoints"]:
    drive_path, local = f"{DRIVE_DIR}/{sub}", f"{REPO_DIR}/{sub}"
    os.makedirs(drive_path, exist_ok=True)
    if os.path.islink(local):
        continue
    if os.path.isdir(local):
        shutil.copytree(local, drive_path, dirs_exist_ok=True)   # copy placeholders (e.g. data/README.md)
        shutil.rmtree(local)
    os.symlink(drive_path, local)
os.makedirs(f"{DRIVE_DIR}/results", exist_ok=True)
!rsync -a {DRIVE_DIR}/results/ results/      # restore runs.csv etc. from earlier sessions
!ls -la data checkpoints

In [ ]:
!python -m src.download_data --glove     # only downloads what is missing (stored on Drive)

### Tests and a 1-minute smoke test

In [ ]:
!python -m pytest
!python -m src.train --model ae_dense --dataset salminen --sample 500 --no-cross

### Full runs
`run_all.sh` runs every experiment in order (EDA → baselines → tuning → autoencoders → hybrid → ablation →
evaluation → explanations → modern fakes). On a T4 it uses the **full** DistilBERT config (`distilbert`), not the
CPU subset. You can also run single models - each line is independent and resumable (DistilBERT keeps a
per-epoch `resume.pt` in its checkpoint folder on Drive).

In [ ]:
# Example: the GPU-only experiments
!python -m src.train --model distilbert --dataset salminen --seeds 42 43 44
# !python -m src.train --model roberta --dataset salminen --seed 42      # optional, if time allows
!rsync -a results/ {DRIVE_DIR}/results/

In [ ]:
# Everything (several hours). Results are synced to Drive at the end; re-run the cell below if the session dies.
!bash run_all.sh
!rsync -a results/ {DRIVE_DIR}/results/

### Save results

In [ ]:
!python -m src.evaluate --tables --plots --no-latent
!rsync -a results/ {DRIVE_DIR}/results/
print("Results are in", DRIVE_DIR + "/results  - copy them into your local repo and commit.")